# Train an emergency-vehicle detector (YOLOv8)

The standard YOLOv8 and Faster R-CNN models are trained on COCO, which has **no ambulance, fire truck or police class**. This notebook fine-tunes YOLOv8 on a labelled emergency-vehicle dataset, so the app can detect them.

**Kaggle settings:** Accelerator → **GPU T4 x2**, Internet → **On**. Training takes about 30–60 minutes.

### Step 0: choose a dataset (free)
1. Create a free account at [Roboflow Universe](https://universe.roboflow.com) and search **"emergency vehicle detection"**. Choose a dataset with bounding boxes for ambulance / fire truck / police (1,500+ images is good) and check its licence.
2. Click **Download Dataset → YOLOv8 → show download code**. Note the workspace name, project name and version number.
3. Copy your **private API key** from Roboflow settings. In Kaggle: **Add-ons → Secrets → Add a new secret**, label `ROBOFLOW_API_KEY`, paste the key. (Never paste the key into a notebook you share.)

In [ ]:
!pip install -q ultralytics roboflow

## 1. Download the dataset

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

WORKSPACE = "PASTE_WORKSPACE"   # from the Roboflow download code
PROJECT = "PASTE_PROJECT"
VERSION = 1

rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location="/kaggle/working/emergency_data")
print(dataset.location)

## 2. Check the classes
The app treats a class as an emergency vehicle when its name contains **ambulance, fire, police** or **emergency**. If the dataset uses other names, rename them in `data.yaml` here.

In [ ]:
import yaml
data_yaml = f"{dataset.location}/data.yaml"
data = yaml.safe_load(open(data_yaml))
print("classes:", data["names"])
keywords = ("ambulance", "fire", "police", "emergency")
print("recognised as emergency:", [n for n in data["names"] if any(k in n.lower() for k in keywords)])

In [ ]:
import glob, os
for split in ("train", "valid", "test"):
    print(split, len(glob.glob(f"{dataset.location}/{split}/images/*")), "images")

## 3. Train
Fine-tune YOLOv8s (pretrained on COCO) for up to 50 epochs; training stops early if validation mAP stops improving for 10 epochs.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(data=data_yaml, epochs=50, imgsz=640, batch=16, patience=10, device=0,
            project="/kaggle/working/runs", name="emergency", exist_ok=True, plots=True)

## 4. Evaluate on the test set (images the model never saw)

In [ ]:
import json
best = YOLO("/kaggle/working/runs/emergency/weights/best.pt")
metrics = best.val(data=data_yaml, split="test" if os.path.isdir(f"{dataset.location}/test") else "val")
results = {"dataset": f"{WORKSPACE}/{PROJECT} v{VERSION}", "mAP50": round(float(metrics.box.map50), 4),
           "mAP50_95": round(float(metrics.box.map), 4),
           "per_class_mAP50_95": {best.names[int(c)]: round(float(m), 4) for c, m in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index])}}
print(json.dumps(results, indent=2))
json.dump(results, open("/kaggle/working/emergency_metrics.json", "w"), indent=2)

In [ ]:
from IPython.display import Image, display
for name in ("results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"):
    path = f"/kaggle/working/runs/emergency/{name}"
    if os.path.exists(path):
        display(Image(path, width=900))

In [ ]:
import shutil
shutil.copy("/kaggle/working/runs/emergency/weights/best.pt", "/kaggle/working/emergency_yolov8s.pt")
print("Download emergency_yolov8s.pt and emergency_metrics.json from the Output tab.")

## Observations
*(Write 3–4 points: test mAP, which class is hardest, typical mistakes you see in the prediction images.)*

## Use it in the app
Put `emergency_yolov8s.pt` in the app's `models/` folder and restart the app. Choose it under **Emergency-vehicle model** on the dashboard: the normal YOLOv8 model keeps counting all vehicles, and this model adds ambulance / fire truck / police detection, alerts and signal priority.